In [ ]:
import numpy as np

# Constants
NGASES = 3  # CH4, O2, CO2

# Henry's law constants
c_h = np.array([1600.0, 1500.0, 2400.0])  # K - CH4, O2, CO2
kh_theta = np.array([1.4e-3, 1.3e-3, 3.4e-2])  # mol/L/atm - CH4, O2, CO2
kh_tbase = 298.15  # K - base temperature

# Gas constant (L*atm/mol/K)
rgasLatm = 0.08206  # You may need to adjust this value based on your application

def henry_law(t_grnd, t_soisno, maxsnl, nl_soil, ngases=NGASES):
    """
    Calculate dimensionless Henry's coefficients for soil gases.
    
    Parameters
    ----------
    t_grnd : float
        Ground surface temperature [K]
    t_soisno : array_like
        Soil temperature array [K], indexed from (maxsnl+1) to nl_soil
    maxsnl : int
        Maximum number of snow layers (negative value)
    nl_soil : int
        Number of soil layers
    ngases : int, optional
        Number of gases (default: 3 for CH4, O2, CO2)
    
    Returns
    -------
    k_h_cc : ndarray
        Dimensionless Henry's coefficient [-]
        Shape: (nl_soil+1, ngases)
        Index 0 corresponds to surface, indices 1:nl_soil+1 correspond to soil layers
    """
    # Initialize output array
    k_h_cc = np.zeros((nl_soil + 1, ngases))
    k_h_inv_last = np.zeros((nl_soil + 1, ngases))
    
    # Loop over soil layers (0 = surface, 1:nl_soil = soil layers)
    for j in range(nl_soil + 1):
        # Loop over gases
        for s in range(ngases):
            if j == 0:
                # Surface layer - use ground temperature
                k_h = kh_theta[s] * np.exp(c_h[s] * (1.0 / t_grnd - 1.0 / kh_tbase))
                k_h_cc[j, s] = k_h * rgasLatm * t_grnd
                k_h_inv_last[j, s] = 1/k_h

            else:
                # Soil layer - use soil temperature
                # Note: t_soisno is indexed from (maxsnl+1) to nl_soil in Fortran
                # In Python (0-indexed), we access t_soisno[j-1] for layer j
                k_h = kh_theta[s] * np.exp(c_h[s] * (1.0 / t_soisno[j - 1] - 1.0 / kh_tbase))
                k_h_cc[j, s] = k_h * rgasLatm * t_soisno[j - 1]
                k_h_inv_last[j, s] = 1/k_h

    
    return k_h_cc,k_h_inv_last


# Example usage
if __name__ == "__main__":
    # Example parameters
    maxsnl = -5  # Example: 5 snow layers (negative by convention)
    nl_soil = 10  # 10 soil layers
    t_grnd = 285.0  # K
    t_soisno = np.linspace(285.0, 280.0, nl_soil)  # Temperature profile
    
    # Calculate Henry's coefficients
    k_h_cc,k_h_inv = henry_law(t_grnd, t_soisno, maxsnl, nl_soil)
    
    print("\n" + "="*60)
    print("Summary of Results:")
    print("="*60)
    print(f"k_h_cc shape: {k_h_cc.shape}")
    print(f"\nSurface (j=0) - Dimensionless Henry's coefficients:")
    print(f"  CH4: {k_h_cc[0, 0]:.6f}")
    print(f"  O2:  {k_h_cc[0, 1]:.6f}")
    print(f"\nFirst soil layer (j=1):")
    print(f"  CH4: {k_h_cc[1, 0]:.6f}")
    print(f"  O2:  {k_h_cc[1, 1]:.6f}")
    
    print(f"\nSurface (j=0) - Henry's constant (inverse) [L*atm/mol]:")
    print(f"  CH4: {k_h_inv[0, 0]:.6f}")
    print(f"  O2:  {k_h_inv[0, 1]:.6f}")

0.9996

Summary of Results:
k_h_cc shape: (11, 3)

Surface (j=0) - Dimensionless Henry's coefficients:
  CH4: 0.041941
  O2:  0.038347

First soil layer (j=1):
  CH4: 0.041941
  O2:  0.038347

Surface (j=0) - Henry's constant (inverse) [L*atm/mol]:
  CH4: 557.618061
  O2:  609.877289
